# Three-Phase SPWM Inverter Model

Run the whole project in your browser. Nothing is installed on your machine and
no account is needed beyond a Google login.

Press **Runtime -> Run all**, or run the cells one at a time. The first cell takes
about twenty seconds; everything after it is quick.

Repository: https://github.com/fatinnihal532-hub/spwm-inverter-model


## Setup


In [ ]:
%cd /content
import os
if not os.path.isdir('spwm-inverter-model'):
    !git clone -q https://github.com/fatinnihal532-hub/spwm-inverter-model.git
%cd spwm-inverter-model
!pip install -q -r requirements.txt
print('ready')


## 1. Does the model agree with theory?

This is the part worth watching. Each check computes a quantity twice: once by
running the simulation, and once from an expression that takes two lines on paper.
If any pair disagrees by more than its tolerance, the script exits non-zero.

Takes about forty seconds.


In [ ]:
!python3 verify.py


## 2. One operating point

A 600 V bus, 50 Hz output, carrier at 1950 Hz, driving a wye-connected RL load.
Change `m` or `scheme` below and run it again. `scheme` takes `"spwm"`,
`"thipwm"` or `"svpwm"`.


In [ ]:
from models.inverter_model import InverterParams, simulate

p = InverterParams(vdc=600, f1=50, fc=1950, m=0.9, scheme='spwm', R=10, L=20e-3)
r = simulate(p, cycles=6)

print(f"line-to-line fundamental   {r['v1_ab_peak']:7.1f} V peak")
print(f"theory  m*Vdc*sqrt(3)/2    {p.m*p.vdc*3**0.5/2:7.1f} V peak")
print(f"voltage THD                {r['thd_vab']*100:7.1f} %")
print(f"current THD                {r['thd_ia']*100:7.2f} %")
print(f"load current rms           {r['ia_rms']:7.2f} A")


### The waveforms that produced those numbers


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

t0, t1 = 3/p.f1, 5/p.f1          # two settled cycles
m = (r['t'] >= t0) & (r['t'] <= t1)
tm = (r['t'][m] - t0) * 1e3

fig, ax = plt.subplots(3, 1, figsize=(9, 7), sharex=True)
ax[0].plot(tm, r['v_ab'][m], lw=0.7)
ax[0].set_ylabel('$v_{ab}$ (V)')
ax[0].set_title('Line-to-line voltage: three levels')
ax[1].plot(tm, r['v_an'][m], lw=0.7, color='tab:green')
ax[1].set_ylabel('$v_{an}$ (V)')
ax[1].set_title('Phase voltage at the floating star point')
for i, lab in enumerate(['a', 'b', 'c']):
    ax[2].plot(tm, r['i'][m, i], lw=1.4, label='phase ' + lab)
ax[2].set_ylabel('current (A)'); ax[2].set_xlabel('time (ms)')
ax[2].set_title('Load current: the inductance has done the filtering')
ax[2].legend(ncol=3)
for a in ax: a.grid(alpha=.3)
plt.tight_layout(); plt.show()


## 3. The result the project exists to show

SPWM runs out of room at a modulation index of 1. Adding a term common to all
three phases -- a third harmonic, or the min-max offset that SVPWM reaches by
another route -- cancels in every line-to-line voltage and cannot push current
through a floating star point. It costs nothing at the load and buys headroom at
the pole, so the linear range extends to 2/sqrt(3) = 1.1547.

Run this and read the last column.


In [ ]:
import numpy as np
from models.inverter_model import InverterParams, simulate

print(f"{'scheme':10s} {'max m':>8s} {'V_ab,1':>10s} {'Vrms/Vdc':>10s} {'THD':>8s}")
rows = {}
for scheme, m_max in (('spwm', 1.0), ('thipwm', 1.1547), ('svpwm', 1.1547)):
    rr = simulate(InverterParams(m=m_max, scheme=scheme), cycles=4)
    util = rr['v1_ab_peak'] / np.sqrt(2) / 600
    rows[scheme] = util
    print(f'{scheme:10s} {m_max:8.4f} {rr["v1_ab_peak"]:9.1f}V {util:10.4f} {rr["thd_vab"]*100:7.1f}%')

gain = 100 * (rows['svpwm'] / rows['spwm'] - 1)
print(f'\ninjection buys {gain:.1f} % more line voltage from the same DC bus')
print(f'theory says   {100*(2/np.sqrt(3)-1):.2f} %')


## 4. Rebuild every figure in the README

Not one of them was drawn by hand. This regenerates all eight from the model,
which takes about ninety seconds.


In [ ]:
!python3 make_figures.py


In [ ]:
from IPython.display import SVG, display
for name in ['scheme_comparison', 'modulation', 'spectrum', 'waveforms']:
    display(SVG(filename=f'figures/{name}_light.svg'))


---

Built by Fatin Nihal Islam. The full write-up, including what the model
deliberately leaves out, is in the
[repository README](https://github.com/fatinnihal532-hub/spwm-inverter-model).
